<a href="https://colab.research.google.com/github/ingkapat/t/blob/main/clean.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 1: Upload dataset.jsonl
from google.colab import files
uploaded = files.upload()  # เลือก dataset.jsonl

Saving dataset.jsonl to dataset.jsonl


In [ ]:
# Cell 2: Remove duplicates and save clean JSONL
import json

filename = list(uploaded.keys())[0]

records = []
with open(filename, encoding='utf-8') as f:
    for line in f:
        records.append(json.loads(line))

def full_text(r):
    return ' '.join(t['text'].strip() for t in r['turns'])

seen = set()
clean = []
for r in records:
    key = full_text(r)
    if key in seen:
        continue
    seen.add(key)
    clean.append(r)

print(f'Before : {len(records)}')
print(f'Removed: {len(records) - len(clean)} duplicates')
print(f'After  : {len(clean)}')

with open('dataset_clean.jsonl', 'w', encoding='utf-8') as f:
    for r in clean:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

Before : 504
Removed: 8 duplicates
After  : 496


In [ ]:
# Cell 3: Convert to Excel for manual audit
import pandas as pd

rows = []
for i, r in enumerate(clean):
    text = ' | '.join(f'[{t["speaker"]}] {t["text"]}' for t in r['turns'])
    rows.append({
        'idx':          i,
        'label':        r['label'],
        'label_text':   'SCAM' if r['label'] == 1 else 'NOT_SCAM',
        'n_turns':      len(r['turns']),
        'text_length':  sum(len(t['text']) for t in r['turns']),
        'text':         text,
        'manual_check': '',   # DELETE หรือ FIX
        'should_be':    '',   # 0 หรือ 1 ถ้า label ผิด
        'note':         '',   # หมายเหตุอื่นๆ
    })

df = pd.DataFrame(rows)
df.to_excel('dataset_review.xlsx', index=False)
print(f'Exported {len(df)} records to dataset_review.xlsx')

Exported 496 records to dataset_review.xlsx


In [ ]:
# Cell 4: Download both files
files.download('dataset_clean.jsonl')   # JSONL ไม่มี duplicate
files.download('dataset_review.xlsx')   # Excel สำหรับ audit

In [ ]:
# Cell 5: (รันหลัง audit เสร็จ) Upload Excel ที่แก้แล้ว → save กลับเป็น JSONL
uploaded2 = files.upload()  # เลือก dataset_review.xlsx ที่แก้แล้ว

df_reviewed = pd.read_excel('dataset_review.xlsx')

final = []
for _, row in df_reviewed.iterrows():
    # ข้ามตัวที่ mark DELETE
    if str(row['manual_check']).strip().upper() == 'DELETE':
        continue

    r = clean[int(row['idx'])]

    # แก้ label ถ้ามี should_be
    if str(row['should_be']).strip() in ('0', '1'):
        r['label'] = int(row['should_be'])

    final.append(r)

print(f'Before audit : {len(clean)}')
print(f'After audit  : {len(final)}')
print(f'Removed      : {len(clean) - len(final)}')

with open('dataset_final.jsonl', 'w', encoding='utf-8') as f:
    for r in final:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

files.download('dataset_final.jsonl')